# Data Preprocessing with Scikit-Learn
A complete pipeline covering:
1. **Missing Value Imputation** — SimpleImputer
2. **Categorical Encoding** — LabelEncoder, OrdinalEncoder, OneHotEncoder
3. **Feature Scaling** — StandardScaler, MinMaxScaler
4. **Train/Test Split** — train_test_split

## Step 1 — Missing Value Imputation
Real datasets often contain `NaN` (missing) values. Two common strategies:
- **Drop rows** with `dropna()` — safe only when you have lots of data
- **Impute** — fill with mean / median / most_frequent using `SimpleImputer`

> `SimpleImputer` is preferred because it integrates into sklearn Pipelines and avoids pandas Copy-on-Write bugs.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer  # sklearn tool to fill missing values

# Sample data — None becomes NaN inside a DataFrame
data = {
    'Name':   ['Ram', 'Shyam', 'Karan', 'Raju', 'Priya'],
    'Age':    [12,    25,      None,    30,     None ],  # 2 missing
    'Salary': [10000, 20000,   None,    None,   15000],  # 2 missing
}
df = pd.DataFrame(data)

print('=== Original DataFrame ===')
print(df)

# isnull() returns True/False mask; .sum() counts Trues per column
print('\n=== Missing Value Count per Column ===')
print(df.isnull().sum())

# isnull().mean() gives proportion (0–1); *100 converts to percentage
print('\n=== Missing Value Percentage ===')
print((df.isnull().mean() * 100).round(1).astype(str) + ' %')

# --- Strategy 1: Drop rows that have ANY NaN ---
# Use when: dataset is large and missing rows are few
df_dropped = df.dropna()
print(f'\n=== After dropna() — {len(df) - len(df_dropped)} row(s) removed ===')
print(df_dropped.reset_index(drop=True))  # reset_index removes the old row numbers

# --- Strategy 2: Impute missing values with column mean ---
# Use when: you want to keep all rows and not lose information
num_cols = ['Age', 'Salary']  # SimpleImputer only works on numeric columns

# strategy='mean' → fills NaN with the average of that column
# Other options: 'median' (better when outliers exist), 'most_frequent' (for categories)
imputer = SimpleImputer(strategy='mean')

df_imputed = df.copy()  # always work on a copy to keep original safe

# fit_transform: fit() learns the mean from data, transform() fills NaN with it
df_imputed[num_cols] = imputer.fit_transform(df[num_cols])

print('\n=== After Mean Imputation (SimpleImputer) ===')
print(df_imputed)

# imputer.statistics_ stores the learned values (mean per column)
print(f'\nImputer learned means → Age: {imputer.statistics_[0]:.1f}, Salary: {imputer.statistics_[1]:.1f}')

=== Original DataFrame ===
    Name   Age   Salary
0    Ram  12.0  10000.0
1  Shyam  25.0  20000.0
2  Karan   NaN      NaN
3   Raju  30.0      NaN
4  Priya   NaN  15000.0

=== Missing Value Count per Column ===
Name      0
Age       2
Salary    2
dtype: int64

=== Missing Value Percentage ===
Name       0.0 %
Age       40.0 %
Salary    40.0 %
dtype: str

=== After dropna() — 3 row(s) removed ===
    Name   Age   Salary
0    Ram  12.0  10000.0
1  Shyam  25.0  20000.0

=== After Mean Imputation (SimpleImputer) ===
    Name        Age   Salary
0    Ram  12.000000  10000.0
1  Shyam  25.000000  20000.0
2  Karan  22.333333  15000.0
3   Raju  30.000000  15000.0
4  Priya  22.333333  15000.0

Imputer learned means → Age: 22.3, Salary: 15000.0


## Step 2 — Categorical Encoding
ML models need numbers — text categories must be converted.

| Encoder | Best For | Output |
|---|---|---|
| **LabelEncoder** | Binary target (Yes/No, Pass/Fail) | 0 or 1 |
| **OrdinalEncoder** | Multiple columns with no order concern | Integer codes |
| **OneHotEncoder** | Nominal categories (city, color — no ranking) | Separate 0/1 columns |

> Never use LabelEncoder on multi-class features — it implies a false ordering (Delhi=0 < Mumbai=1).

In [2]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

df_enc = pd.DataFrame({
    'Name':   ['Amit', 'Priya', 'Rahul', 'Sneha', 'Vikas'],
    'Gender': ['Male', 'Female', 'Male', 'Female', 'Male'],
    'City':   ['Delhi', 'Mumbai', 'Pune', 'Delhi', 'Kolkata'],
    'Passed': ['Yes', 'Yes', 'No', 'Yes', 'No'],
})

print('=== Original Data ===')
print(df_enc)

# --- LabelEncoder: converts 1 column to integer codes ---
# Best for binary target columns (Yes/No, True/False)
le = LabelEncoder()

# fit_transform: learns unique classes then maps each to 0,1,2...
df_enc['Passed_Encoded'] = le.fit_transform(df_enc['Passed'])

# le.classes_ shows what the encoder found in sorted order
print('\n=== LabelEncoder on "Passed" ===')
print(f'Classes found: {list(le.classes_)} → mapped to 0, 1')  # No=0, Yes=1 (alphabetical)
print(df_enc[['Name', 'Passed', 'Passed_Encoded']])

# --- OrdinalEncoder: handles MULTIPLE columns at once ---
# Assigns integers based on sorted order of unique values
oe = OrdinalEncoder()

# Pass a 2D array (list of columns) — returns same shape with integer codes
df_enc[['Gender_Encoded', 'City_Encoded']] = oe.fit_transform(df_enc[['Gender', 'City']])

print('\n=== OrdinalEncoder on Gender + City ===')
# oe.categories_ is a list of arrays — one per column
print('Gender categories:', oe.categories_[0].tolist())  # ['Female', 'Male']
print('City categories:  ', oe.categories_[1].tolist())  # sorted alphabetically
print(df_enc[['Name', 'Gender', 'Gender_Encoded', 'City', 'City_Encoded']])

# --- OneHotEncoder: creates one binary column per unique category ---
# Best for nominal (unordered) features like City — avoids false ranking
ohe = OneHotEncoder(sparse_output=False)  # sparse_output=False → returns plain numpy array

# fit_transform expects 2D input, hence [['City']] instead of ['City']
city_encoded = ohe.fit_transform(df_enc[['City']])

# get_feature_names_out gives column names like City_Delhi, City_Mumbai...
city_cols = ohe.get_feature_names_out(['City'])

# Combine original columns with the new one-hot columns side by side
df_ohe = pd.concat(
    [df_enc[['Name', 'City']].reset_index(drop=True),
     pd.DataFrame(city_encoded, columns=city_cols)],
    axis=1  # axis=1 means join column-wise (side by side)
)
print('\n=== OneHotEncoder on "City" (1 column per city) ===')
print(df_ohe)

=== Original Data ===
    Name  Gender     City Passed
0   Amit    Male    Delhi    Yes
1  Priya  Female   Mumbai    Yes
2  Rahul    Male     Pune     No
3  Sneha  Female    Delhi    Yes
4  Vikas    Male  Kolkata     No

=== LabelEncoder on "Passed" ===
Classes found: ['No', 'Yes'] → mapped to 0, 1
    Name Passed  Passed_Encoded
0   Amit    Yes               1
1  Priya    Yes               1
2  Rahul     No               0
3  Sneha    Yes               1
4  Vikas     No               0

=== OrdinalEncoder on Gender + City ===
Gender categories: ['Female', 'Male']
City categories:   ['Delhi', 'Kolkata', 'Mumbai', 'Pune']
    Name  Gender  Gender_Encoded     City  City_Encoded
0   Amit    Male             1.0    Delhi           0.0
1  Priya  Female             0.0   Mumbai           2.0
2  Rahul    Male             1.0     Pune           3.0
3  Sneha  Female             0.0    Delhi           0.0
4  Vikas    Male             1.0  Kolkata           1.0

=== OneHotEncoder on "City" (1 col

## Step 3 — Feature Scaling
Algorithms like KNN, SVM, and neural networks are sensitive to feature magnitude.
Scaling ensures no single feature dominates just because of its unit.

| Scaler | Formula | Range | Use When |
|---|---|---|---|
| **StandardScaler** | (x − μ) / σ | Mean=0, Std=1 | Normal-ish distribution |
| **MinMaxScaler** | (x − min) / (max − min) | [0, 1] | Bounded data, no outliers |

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df_scale = pd.DataFrame({
    'StudyHours': [1, 2, 3, 4, 5],
    'TestScore':  [40, 55, 60, 72, 80],
    'Attendance': [60, 70, 80, 90, 95],
})

print('=== Original Data ===')
print(df_scale)
print('\nStats before scaling:')
print(df_scale.describe().round(2))

# --- StandardScaler: removes mean and scales to unit variance ---
# After scaling: mean ≈ 0, std ≈ 1 for every column
std_scaler = StandardScaler()

# fit_transform = fit() (learn mean+std) + transform() (apply formula)
std_scaled = std_scaler.fit_transform(df_scale)

# Wrap back in DataFrame to keep column names readable
df_std = pd.DataFrame(std_scaled, columns=df_scale.columns)
print('\n=== StandardScaler Output (mean≈0, std≈1) ===')
print(df_std.round(4))

# std_scaler.mean_ stores the mean of each column (learned during fit)
print(f'\nLearned column means   : {dict(zip(df_scale.columns, std_scaler.mean_.round(2)))}')
# std_scaler.scale_ stores the std deviation of each column
print(f'Learned column std devs: {dict(zip(df_scale.columns, std_scaler.scale_.round(2)))}')

# --- MinMaxScaler: compresses values into [0, 1] range ---
# Formula: (x - min) / (max - min)
mm_scaler = MinMaxScaler()

mm_scaled = mm_scaler.fit_transform(df_scale)  # fit learns min/max, transform applies them
df_mm = pd.DataFrame(mm_scaled, columns=df_scale.columns)

print('\n=== MinMaxScaler Output (range [0.0 – 1.0]) ===')
print(df_mm.round(4))

# data_min_ and data_max_ store the observed min/max per column
print(f'\nData min per column: {dict(zip(df_scale.columns, mm_scaler.data_min_.astype(int)))}')
print(f'Data max per column: {dict(zip(df_scale.columns, mm_scaler.data_max_.astype(int)))}')

=== Original Data ===
   StudyHours  TestScore  Attendance
0           1         40          60
1           2         55          70
2           3         60          80
3           4         72          90
4           5         80          95

Stats before scaling:
       StudyHours  TestScore  Attendance
count        5.00       5.00        5.00
mean         3.00      61.40       79.00
std          1.58      15.49       14.32
min          1.00      40.00       60.00
25%          2.00      55.00       70.00
50%          3.00      60.00       80.00
75%          4.00      72.00       90.00
max          5.00      80.00       95.00

=== StandardScaler Output (mean≈0, std≈1) ===
   StudyHours  TestScore  Attendance
0     -1.4142    -1.5451     -1.4837
1     -0.7071    -0.4621     -0.7028
2      0.0000    -0.1011      0.0781
3      0.7071     0.7653      0.8590
4      1.4142     1.3429      1.2494

Learned column means   : {'StudyHours': np.float64(3.0), 'TestScore': np.float64(61.4), 'Atten

## Step 4 — Train / Test Split
Always split data **before** fitting any scaler or imputer.
Fitting on the full dataset causes **data leakage** — the model indirectly "sees" test data.

> Rule: `fit_transform` on **train set** → `transform` only on **test set**.

In [4]:
from sklearn.model_selection import train_test_split

# Synthetic dataset — larger so the split is meaningful
np.random.seed(42)  # fixed seed → reproducible random results every run
n = 20
df_ml = pd.DataFrame({
    'StudyHours': np.round(np.random.uniform(1, 10, n), 1),  # float between 1–10
    'Attendance': np.random.randint(50, 100, n),              # int between 50–100
    'TestScore':  np.random.randint(40, 100, n),              # target variable
})

X = df_ml[['StudyHours', 'Attendance']]  # features (inputs to the model)
y = df_ml['TestScore']                   # target  (what we want to predict)

# test_size=0.2 → 20% goes to test, 80% to train
# random_state=42 → shuffle is reproducible (same split every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Total samples : {len(df_ml)}')
print(f'Training set  : {len(X_train)} samples ({len(X_train)/len(df_ml)*100:.0f}%)')
print(f'Test set      : {len(X_test)} samples  ({len(X_test)/len(df_ml)*100:.0f}%)')

# --- Scale AFTER splitting ---
# IMPORTANT: scaler must only learn from training data (no peeking at test)
scaler = StandardScaler()

# fit_transform on train: learns mean+std FROM training data, then scales it
X_train_scaled = scaler.fit_transform(X_train)

# transform only on test: uses the SAME mean+std learned from train (not test)
# This simulates real-world: in production you only have train stats available
X_test_scaled = scaler.transform(X_test)

print('\n=== Training Features (scaled) ===')
print(pd.DataFrame(X_train_scaled, columns=X.columns).round(3))

print('\n=== Test Features (scaled using train mean/std) ===')
print(pd.DataFrame(X_test_scaled, columns=X.columns).round(3))

print('\n=== Training Labels (y_train) ===')
print(y_train.values)

print('\n=== Test Labels (y_test) ===')
print(y_test.values)

Total samples : 20
Training set  : 16 samples (80%)
Test set      : 4 samples  (20%)

=== Training Features (scaled) ===
    StudyHours  Attendance
0        0.513      -0.991
1       -0.952       1.347
2        1.722       1.031
3        0.513      -0.486
4       -0.037       0.209
5       -0.476       1.726
6       -0.769      -1.181
7        0.952      -0.423
8        0.879      -0.107
9       -0.513      -1.308
10      -0.952       1.537
11       1.282      -0.296
12       1.392       0.905
13      -1.392      -0.865
14      -0.879       0.146
15      -1.282      -1.244

=== Test Features (scaled using train mean/std) ===
   StudyHours  Attendance
0      -0.220       1.221
1       0.256      -0.865
2      -0.842      -0.549
3       1.685       0.336

=== Training Labels (y_train) ===
[74 83 75 99 81 45 79 86 53 43 46 89 86 56 43 47]

=== Test Labels (y_test) ===
[59 93 41 67]


## Quick Reference — Preprocessing Decision Tree

```
Raw Data
  │
  ├─ Has missing values?
  │    ├─ Many rows available  ──► dropna()
  │    └─ Cannot afford to drop ──► SimpleImputer(strategy='mean'|'median'|'most_frequent')
  │
  ├─ Has categorical columns?
  │    ├─ Binary (Yes/No)      ──► LabelEncoder
  │    ├─ Multiple columns     ──► OrdinalEncoder
  │    └─ Nominal (no order)   ──► OneHotEncoder
  │
  ├─ Numeric features need scaling?
  │    ├─ Normal distribution  ──► StandardScaler  (mean=0, std=1)
  │    └─ Bounded, no outliers ──► MinMaxScaler    (range 0–1)
  │
  └─ Split data
       train_test_split(X, y, test_size=0.2, random_state=42)
       → fit_transform(X_train)   # learn stats from train only
       → transform(X_test)        # apply same stats to test
```